## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when,
    lpad,
    current_date
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_tickets")
silver_customers = spark.table("silver_customers")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

ticket_id,customer_id,ticket_created_at,ticket_resolved_at,ticket_status,priority,channel,category,agent_id,source_system
TCK000000001,CRM0077019,2024-03-19T16:21:56.329Z,null,IN_PROGRESS,LOW,EMAIL,PAYMENT_ISSUE,AGT00401,zendesk
TCK000000002,CRM0021746,2026-02-25T16:21:56.329Z,null,IN_PROGRESS,HIGH,EMAIL,REFUND_REQUEST,AGT00499,zendesk
TCK000000003,CRM0021746,2023-10-10T16:21:56.329Z,2023-10-11T12:21:56.329Z,CLOSED,LOW,WEB,DELIVERY_DELAY,AGT00173,zendesk
TCK000000004,CRM0021746,2023-08-14T16:21:56.329Z,2023-08-21T01:21:56.329Z,RESOLVED,HIGH,WEB,COMPLAINT,AGT00208,zendesk
TCK000000005,CRM0034127,2024-04-21T16:21:56.329Z,null,IN_PROGRESS,HIGH,EMAIL,COMPLAINT,AGT00146,zendesk
TCK000000006,CRM0034127,2025-10-22T16:21:56.329Z,null,IN_PROGRESS,MEDIUM,PHONE,PAYMENT_ISSUE,AGT00202,zendesk
TCK000000007,CRM0060907,2024-07-26T16:21:56.329Z,null,OPEN,HIGH,WEB,PRODUCT_QUERY,AGT00339,zendesk
TCK000000008,CRM0032176,2023-07-24T16:21:56.329Z,null,OPEN,LOW,EMAIL,REFUND_REQUEST,AGT00226,zendesk
TCK000000009,CRM0014917,2025-09-11T16:21:56.329Z,2025-09-16T17:21:56.329Z,RESOLVED,URGENT,CHAT,REFUND_REQUEST,AGT00068,zendesk
TCK000000010,CRM0014917,2024-04-28T16:21:56.329Z,null,OPEN,HIGH,CHAT,LOYALTY_QUERY,AGT00315,zendesk


Bronze rows: 75,857


## Silver transformation

In [0]:
TICKET_STATUSES = ["OPEN", "IN_PROGRESS", "RESOLVED", "CLOSED"]

PRIORITIES = ["LOW", "MEDIUM", "HIGH", "URGENT"]
CHANNELS = ["EMAIL", "PHONE", "CHAT", "WEB"]
CATEGORIES = [
    "PAYMENT_ISSUE",
    "DELIVERY_DELAY",
    "ACCOUNT_ACCESS",
    "REFUND_REQUEST",
    "PRODUCT_QUERY",
    "LOYALTY_QUERY",
    "COMPLAINT"
]

standardised_df = (
    bronze_df
    .withColumn("ticket_created_at", to_date(col("ticket_created_at")))
    .withColumn("ticket_resolved_at", to_date(col("ticket_resolved_at")))
    .withColumn("ticket_status", upper(trim(col("ticket_status"))))
    .withColumn("priority", upper(trim(col("priority"))))
    .withColumn("channel", upper(trim(col("channel"))))
    .withColumn("category", upper(trim(col("category"))))
    .withColumn("_silver_processed_at", current_timestamp())
)
window_spec = Window.partitionBy("ticket_id").orderBy(col("ticket_created_at").desc_nulls_last())
deduped_df = (
    standardised_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") ==1)
    .drop("row_num")
)
customer_lookup = (
    silver_customers
    .select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_exists", lit(True))
)
support_tickets_with_customer_check = (
    deduped_df
    .join(
        customer_lookup,
        on="customer_id",
        how= "left"
    )
)
quality_df = (
    support_tickets_with_customer_check
    .withColumn(
        "_has_future_ticket_date",
        (
            (col("ticket_created_at") > current_date()) | 
            ((col("ticket_resolved_at").isNotNull()) & (col("ticket_resolved_at") >current_date()))
        )
    )
    .withColumn(
        "_is_valid",
        when(col("ticket_id").isNull(), lit(False))
        .when(col("customer_id").isNull(), lit(False))
        .when(col("customer_exists").isNull(), lit(False))
        .when(col("ticket_created_at").isNull(), lit(False))
        .when((col("ticket_resolved_at").isNotNull()) & (col("ticket_resolved_at") < col("ticket_created_at")),lit(False))
        .when(col("ticket_status").isNull(), lit(False))            
        .when(~col("ticket_status").isin(TICKET_STATUSES), lit(False))
        .when(col("priority").isNull(), lit(False))
        .when(~col("priority").isin(PRIORITIES), lit(False))
        .when(col("channel").isNull(), lit(False))
        .when(~col("channel").isin(CHANNELS), lit(False))
        .when(col("category").isNull(), lit(False))
        .when(~col("category").isin(CATEGORIES), lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("ticket_id").isNull(), lit("ticket_id is null"))
        .when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(col("customer_exists").isNull(), lit("customer_id does not exist in silvercustomers"))
        .when(col("ticket_created_at").isNull(), lit("ticket_created_at is null"))
        .when((col("ticket_resolved_at").isNotNull()) & (col("ticket_resolved_at") < col("ticket_created_at")), lit("negative resolution time"))
        .when(col("ticket_status").isNull(), lit("ticket_status is null"))            
        .when(~col("ticket_status").isin(TICKET_STATUSES), lit("invalid ticket_status"))
        .when(col("priority").isNull(), lit("priority is null"))
        .when(~col("priority").isin(PRIORITIES), lit("invalid priority"))
        .when(col("channel").isNull(), lit("channel is null"))
        .when(~col("channel").isin(CHANNELS), lit("invalid channel"))
        .when(col("category").isNull(), lit("category is null"))
        .when(~col("category").isin(CATEGORIES), lit("invalid category"))
        .otherwise(lit(None))
    )
)
valid_support_tickets = quality_df.filter(col("_is_valid")== True)
rejected_support_tickets = quality_df.filter(col("_is_valid") == False)

## Delta tables

In [0]:
(
    valid_support_tickets
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_tickets")
)

(
    rejected_support_tickets
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_tickets")
)

print("Silver Tickets completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid tickets: {valid_support_tickets.count():,}")
print(f"Rejected tickets: {rejected_support_tickets.count():,}")

Silver Tickets completed
Bronze rows: 75,857
After deduplication: 75,106
Valid tickets: 73,623
Rejected tickets: 1,483


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_addresses|      false|
| default|  bronze_customers|      false|
| default|    bronze_loyalty|      false|
| default|     bronze_orders|      false|
| default|   bronze_payments|      false|
| default|    bronze_tickets|      false|
| default|rejected_addresses|      false|
| default|rejected_customers|      false|
| default|  rejected_loyalty|      false|
| default|   rejected_orders|      false|
| default| rejected_payments|      false|
| default|  rejected_tickets|      false|
| default|  silver_addresses|      false|
| default|  silver_customers|      false|
| default|    silver_loyalty|      false|
| default|     silver_orders|      false|
| default|   silver_payments|      false|
| default|    silver_tickets|      false|
+--------+------------------+-----------+

